Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.complete_data"
silver_table = f"{catalog}.{silver_schema}.complete_data"


Read bronze Delta table into a DataFrame

In [0]:
from pyspark.sql import functions as F

bronze_complete_df=(
    spark.read
    .format("delta")
    .table(bronze_table)
    .filter(
        F.col("batch_id") == batch_id
    )
)

Drop null records

In [0]:
bronze_complete_df=bronze_complete_df.dropna()

Drop duplicated records

In [0]:
bronze_complete_df=bronze_complete_df.dropDuplicates()

Transforming fl_date column to integer data type to use as a date_key column

In [0]:
bronze_complete_df=(
    bronze_complete_df
    .withColumn(
        "fl_date",F.date_format("fl_date", "yyyyMMdd").cast("int")
    )
)

Select and rename columns for clarity and unification

In [0]:
silver_complete_df = (
    bronze_complete_df
    .select(
        F.col("fl_date").alias("date_key"),
        F.col("dep_time").alias("departure_timestamp"),
        F.col("mkt_unique_carrier").alias("mkt_unique_carrier_key"),
        "mkt_carrier_fl_num",
        F.col("op_unique_carrier").alias("op_unique_carrier_key"),
        "op_carrier_fl_num",
        F.col("tail_num").alias("tail_number_key"),
        F.col("origin").alias("origin_key"),
        F.col("dest").alias("destination_key"),
        "crs_dep_time",
        "taxi_out",
        "dep_delay",
        "air_time",
        "distance",
        F.col("cancelled").alias("cancellation_key"),
        "latitude",
        "longitude",
        "elevation",
        "mesonet_station",
        F.col("year_of_manufacture").alias("aircraft_year_of_manufacture"),
        F.col("manufacturer").alias("aircraft_manufacturer"),
        F.col("icao_type").alias("aircraft_type"),
        F.col("range").alias("aircraft_range"),
        F.col("width").alias("aircraft_width"),
        "temperature",
        "dew_point",
        "rel_humidity",
        "altimeter",
        "batch_id"
    )
)


Add created and updated timestamp columns

In [0]:
silver_complete_df=(
    silver_complete_df
    .withColumns({
        "created_timestamp":F.current_timestamp(),
        "updated_timestamp":F.current_timestamp()
    })
)

Write DataFrame to silver Delta table 

Subsequent runs merge new batch into existing table, only updating records from a newer or equal batch to avoid reprocessing

In [0]:
if not spark.catalog.tableExists(silver_table):
    silver_complete_df_write = (
    silver_complete_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
    )

else:
    
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            silver_complete_df.alias("s"),
            """t.date_key = s.date_key 
            AND t.mkt_carrier_fl_num = s.mkt_carrier_fl_num 
            AND t.origin_key = s.origin_key 
            AND t.destination_key = s.destination_key
            AND t.tail_number_key = s.tail_number_key
            """
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "departure_timestamp": "s.departure_timestamp",
                "mkt_unique_carrier_key": "s.mkt_unique_carrier_key",
                "mkt_carrier_fl_num": "s.mkt_carrier_fl_num",
                "op_unique_carrier_key": "s.op_unique_carrier_key",
                "op_carrier_fl_num": "s.op_carrier_fl_num",
                "tail_number_key": "s.tail_number_key",
                "origin_key": "s.origin_key",
                "destination_key": "s.destination_key",
                "crs_dep_time": "s.crs_dep_time",
                "taxi_out": "s.taxi_out",
                "dep_delay": "s.dep_delay",
                "air_time": "s.air_time",
                "distance": "s.distance",
                "cancellation_key": "s.cancellation_key",
                "latitude": "s.latitude",
                "longitude": "s.longitude",
                "elevation": "s.elevation",
                "mesonet_station": "s.mesonet_station",
                "aircraft_year_of_manufacture": "s.aircraft_year_of_manufacture",
                "aircraft_manufacturer": "s.aircraft_manufacturer",
                "aircraft_type": "s.aircraft_type",
                "aircraft_range": "s.aircraft_range",
                "aircraft_width": "s.aircraft_width",
                "temperature": "s.temperature",
                "dew_point": "s.dew_point",
                "rel_humidity": "s.rel_humidity",
                "altimeter": "s.altimeter",
                "batch_id": "s.batch_id",
                "updated_timestamp": "s.updated_timestamp"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )